# Single-source contextual decomposition on GPT-2

This notebook demonstrates nonlinear CD allocation from one attention-head position to downstream internal activations and a logit difference. A small randomly initialized GPT-2 runs entirely on CPU, without a tokenizer, Hub download, or external artifact. The model validates the mechanics, not a linguistic mechanism.

Each activation is represented by beta + gamma. Reconstruction, correctness of the allocation equations, and causal faithfulness are different claims. A small reconstruction error does not establish that a source is necessary or sufficient for behavior.

In [1]:
import torch
from transformers import GPT2Config, GPT2LMHeadModel
from transformer_lens.model_bridge.sources import build_bridge_from_module
from transformer_lens.tools.analysis import (
    CDBaseline, CDSource, CDTarget, ContextualDecomposition,
)

config = GPT2Config(vocab_size=16, n_positions=8, n_embd=8, n_layer=2, n_head=2)
config._attn_implementation = 'eager'
with torch.random.fork_rng():
    torch.manual_seed(5)
    model = build_bridge_from_module(
        GPT2LMHeadModel(config).eval(), 'GPT2LMHeadModel', hf_config=config,
        dtype=torch.float32, device='cpu',
    ).eval()
tokens = torch.tensor([[1, 2, 3], [3, 2, 1]])
analysis = ContextualDecomposition(model)
print('Raw fp32 eager GPT-2 initialized locally.')

Raw fp32 eager GPT-2 initialized locally.


## Read one head-position contribution

The source is head 1 at position 0 in layer 0. Its residual-space write excludes the shared attention-output bias. A zero reference assigns that write to beta and the remaining native residual to gamma at the attention merge. All downstream positions remain in the traversal.

The two examples use different target positions and token IDs. Results retain one signed logit-difference value per example; they are not automatically averaged.

In [2]:
source = CDSource.attention_head(0, 1, positions=(0,))
target = CDTarget.logit_difference(
    positive_token_id=(2, 3), negative_token_id=(1, 2), position=(1, 2),
)
result = analysis.decompose(
    tokens, source=source, target=target, baseline=CDBaseline.zero(),
    retain_intermediates=('blocks.1.ln1.hook_out',),
)
assert result.target_beta.shape == (2,)
assert not result.target_beta.requires_grad
for diagnostic in result.diagnostics.values():
    assert diagnostic.max_abs <= diagnostic.tolerance
assert torch.isfinite(result.target_beta).all()
print('Per-example logit differences reconstruct within the native error budget.')

Per-example logit differences reconstruct within the native error budget.


## Inspect an internal tensor

The retained ln1 tensor has shape [batch, sequence, width]. Its beta/gamma and independent native total can be inspected without retaining every intermediate in the result. This retention option does not bound peak executor memory. A separate internal target can select a head in Q/K/V or attention-context coordinates.

In [3]:
internal = result.intermediates['blocks.1.ln1.hook_out']
assert internal.beta.shape == (2, 3, 8)
q_result = analysis.decompose(
    tokens, source=source,
    target=CDTarget.internal('blocks.1.attn.q.hook_out', position=(1, 2), head=0),
)
assert q_result.target_beta.shape == (2, 4)
assert not q_result.intermediates
assert q_result.diagnostics['target'].max_abs <= q_result.diagnostics['target'].tolerance
print('Internal residual and per-head target spaces are explicit.')

Internal residual and per-head target spaces are explicit.


## Supplied reference and a no-op source check

A reference tensor must be finite fp32 [batch, sequence, width] on the model device. CDBaseline snapshots it. Here a final residual source is compared with its identical native reference, so its beta is zero. This checks reference and anchoring mechanics; it does not test an attention intervention.

In [4]:
residual_source = CDSource.residual(1, location='post', positions=(2,))
with torch.no_grad():
    _, native_cache = model.run_with_cache(tokens)
reference = CDBaseline.from_tensor(
    native_cache[residual_source.anchor], description='same native residual',
)
no_op = analysis.decompose(
    tokens, source=residual_source, target=CDTarget.logit(token_id=(2, 3), position=2),
    baseline=reference,
)
assert torch.equal(no_op.target_beta, torch.zeros_like(no_op.target_beta))
print('A native-equal source reference produces zero assigned beta.')

A native-equal source reference produces zero assigned beta.


## Verify repeatability and state preservation

Read-only observers and existing gradients are preserved. Editing hooks, active relevance rules, unsupported architecture operations, and fully masked causal rows are rejected. No gradients are computed by CD.

In [5]:
versions = {name: parameter._version for name, parameter in model.named_parameters()}
rng = torch.random.get_rng_state().clone()
repeat = analysis.decompose(tokens, source=source, target=target)
torch.testing.assert_close(repeat.target_beta, result.target_beta)
assert versions == {name: parameter._version for name, parameter in model.named_parameters()}
assert torch.equal(torch.random.get_rng_state(), rng)
assert all(parameter.grad is None for parameter in model.parameters())
print('Repeated analysis preserves weights, RNG, and gradient state.')

Repeated analysis preserves weights, RNG, and gradient state.


## Interpretation, numerical limits, and trained models

The policy magnitude-affine-ordered-gelu-v1 allocates affine bias by projected magnitude, uses ordered GELU differences, shares the full-input LayerNorm denominator, and decomposes nonlinear attention. It is not a symmetric Shapley allocation or an LRP rule. Negative shares are valid. Independent scalar tests verify the constituent equations; native reconstruction alone cannot do so.

The numerical budget is 2e-6 + 2e-5 times the maximum native tensor magnitude, not an elementwise relative threshold at near-zero coordinates. Logit differences propagate input-logit budgets. Constituent cancellation magnitude is reported but does not enlarge the budget. Excess errors raise rather than correcting gamma.

For trained characterization, load gpt2 with revision 607a30d783dfa663caf39e06633721c8d4cfcd7e, CPU/fp32/eager attention, explicit input IDs [15496, 995, 0], and no implicit BOS. That experiment is intentionally outside this notebook's no-download path; source selection, masks, references, and target IDs must be recorded. Use model task capability and actual interventions before making mechanism claims.

Single-source analysis does not imply joint-source attribution, dataset ranking, automated circuit discovery, or support for every GPT-2 variant. See the contextual-decomposition guide for restrictions and API details.